# Dataset 6 — Cleaning
**Danish adolescents — produces ds6_behavioral_clean.csv, ds6_survey_clean.csv, ds6_demographics_clean.csv, ds6_participant_linkage.csv**

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

RAW6 = Path("../data_raw/6. data set, Danish adolescents/Data")
SESSION  = RAW6 / "behavioral data/session_time_dataset_97_5_cutoff_data_forwarded.csv"
DEMO     = RAW6 / "behavioral data/demographic_data.csv"
SURVEY_C = RAW6 / "behavioral data/survey_data_clean.csv"
OUT = Path("../outputs/cleaned")
OUT.mkdir(parents=True, exist_ok=True)
print("Paths configured")

## 1. Load Raw Files

In [ ]:
sess   = pd.read_csv(SESSION,  low_memory=False)
demo   = pd.read_csv(DEMO,     low_memory=False)
surv_c = pd.read_csv(SURVEY_C, low_memory=False)
print(f"Session: {sess.shape}")
print(f"Demo: {demo.shape}")
print(f"Survey clean: {surv_c.shape}")

## 2. Behavioral — Canonical ID

In [ ]:
sess['participant_id_raw'] = sess['ID']
sess['participant_id_canonical'] = sess['ID']
print(f"Unique participants: {sess['participant_id_canonical'].nunique()}")
print("Sample IDs:", sess['participant_id_canonical'].head(3).tolist())

## 3. Behavioral — Timestamp Parsing

In [ ]:
sess['event_timestamp_utc'] = pd.to_datetime(sess['timestamp'], utc=True, errors='coerce')
sess['event_date'] = pd.to_datetime(sess['time'], utc=True, errors='coerce').dt.date
parse_fail = sess['event_timestamp_utc'].isna().sum()
print(f"Parse failures: {parse_fail}")
print("Sample UTC:", sess['event_timestamp_utc'].head(3).tolist())

## 4. Behavioral — Event Outcome

In [ ]:
# All rows are openedApp
print("resolution unique:", sess['resolution'].value_counts(dropna=False).to_dict())
sess['event_outcome_raw'] = sess['resolution']
sess['event_outcome_simple'] = 'opened'
print("event_outcome_simple: all 'opened'")

## 5. Behavioral — Intervention Fields

In [ ]:
sess['intervention_duration_sec'] = pd.to_numeric(sess['interventionDuration'], errors='coerce')
sess['intervention_type']         = sess['interventionType']
sess['is_reintervention']         = pd.to_numeric(sess['isReIntervention'], errors='coerce').fillna(0).astype(int)
sess['reintervention_delay_sec']  = pd.to_numeric(sess['timeIntervalUntilReIntervention'], errors='coerce')

print("is_reintervention:", sess['is_reintervention'].value_counts().to_dict())
print("intervention_type:", sess['intervention_type'].value_counts(dropna=False).head(5).to_dict())

## 6. Behavioral — Study Design Fields

In [ ]:
sess['study_condition_num'] = pd.to_numeric(sess['cond'], errors='coerce')
sess['study_condition']     = sess['condition']
sess['study_period']        = sess['period']
sess['participation_day']   = pd.to_numeric(sess['participation_day'], errors='coerce')
sess['study_week']          = pd.to_numeric(sess['week'], errors='coerce')
# F111: raw `week_day` is a composite 'week_dayofmonth' string (e.g. '40_4'), so
# pd.to_numeric coerced every one of the 353,796 rows to NaN. Splitting the string would
# not repair it either: the second component is day-of-MONTH (identical to the raw `day`
# column, values 1..31), not day-of-week. Derive it from the timestamp instead.
sess['study_day_of_week']   = sess['event_timestamp_utc'].dt.dayofweek  # UTC, Monday=0 .. Sunday=6
assert sess['study_day_of_week'].isna().mean() < 0.01, 'study_day_of_week parse failed'
sess['hour_of_day']         = pd.to_numeric(sess['hour'], errors='coerce')

print("study_period:", sess['study_period'].value_counts().to_dict())
print("study_condition:", sess['study_condition'].value_counts().to_dict())

## 7. Behavioral — days_since_first_event

In [ ]:
first_ev = sess.sort_values('event_timestamp_utc').groupby('participant_id_canonical')['event_timestamp_utc'].transform('min')
sess['days_since_first_event'] = (sess['event_timestamp_utc'] - first_ev).dt.total_seconds() / 86400
print(f"days_since_first_event: min={sess['days_since_first_event'].min():.1f}, max={sess['days_since_first_event'].max():.1f}")

## 8. Behavioral — Select Columns

In [ ]:
sess.rename(columns={'app': 'app_name'}, inplace=True)

COLS = [
    'participant_id_raw', 'participant_id_canonical',
    'event_timestamp_utc', 'event_date',
    'app_name', 'event_outcome_raw', 'event_outcome_simple',
    'study_condition', 'study_condition_num', 'study_period',
    'participation_day', 'study_week', 'study_day_of_week', 'hour_of_day',
    'days_since_first_event',
    'intervention_duration_sec', 'intervention_type',
    'is_reintervention', 'reintervention_delay_sec',
    'group_id', 'app_next', 'app_next2', 'app_next3',
    'app_prev', 'app_prev2',
    'resolution_next', 'resolution_next2', 'resolution_next3',
    'time_diff_secs', 'time_diff_minutes',
    'first_in_group', 'last_close_in_group',
    'purpose', 'row_id',
]
COLS = [c for c in COLS if c in sess.columns]
beh_clean = sess[COLS].copy()
print(f"Behavioral clean: {beh_clean.shape}")

## 9. Behavioral — Validation

In [ ]:
assert beh_clean.duplicated().sum() == 0, "Duplicate rows found!"
assert beh_clean['participant_id_canonical'].notna().all()
assert (beh_clean['event_outcome_simple'] == 'opened').all()
print("All validations passed.")

## 10. Save Behavioral

In [ ]:
beh_clean.to_csv(OUT / "ds6_behavioral_clean.csv", index=False)
print(f"Saved: ds6_behavioral_clean.csv  ({beh_clean.shape})")

## 11. Demographics — Clean

In [ ]:
demo['participant_id_canonical'] = demo['ID']
demo['gender_raw'] = demo['gender']
demo['gender'] = demo['gender'].map({'kvinde': 'female', 'mand': 'male'}).fillna(demo['gender'])
demo['age'] = pd.to_numeric(demo['age'], errors='coerce')

print(f"Demo shape: {demo.shape}")
print("gender:", demo['gender'].value_counts(dropna=False).to_dict())
print("age:", demo['age'].value_counts(dropna=False).sort_index().to_dict())

## 12. Save Demographics

In [ ]:
demo.to_csv(OUT / "ds6_demographics_clean.csv", index=False)
print(f"Saved: ds6_demographics_clean.csv  ({demo.shape})")

## 13. Survey Clean — Numeric Parsing

In [ ]:
surv_c['participant_id_canonical'] = surv_c['response_id']
surv_c['survey_start_dt'] = pd.to_datetime(surv_c['start_date'], errors='coerce')
surv_c['survey_end_dt']   = pd.to_datetime(surv_c['end_date'],   errors='coerce')

for col in ['addiction_index','retention_index','self_control_index','trivsel',
            'socialconnection','fo_mo','kontrolover_so_me','taenkerpa_so_me',
            'tilfredsmed_so_me','survey_nr']:
    surv_c[col] = pd.to_numeric(surv_c[col], errors='coerce')

print(f"Survey clean: {surv_c.shape}")
print("survey_nr:", surv_c['survey_nr'].value_counts(dropna=False).sort_index().to_dict())

## 14. Save Survey

In [ ]:
surv_c.to_csv(OUT / "ds6_survey_clean.csv", index=False)
print(f"Saved: ds6_survey_clean.csv  ({surv_c.shape})")

## 15. Participant Linkage

In [ ]:
beh_ids  = set(beh_clean['participant_id_canonical'].unique())
demo_ids = set(demo['participant_id_canonical'].unique())
surv_ids = set(surv_c['participant_id_canonical'].unique())
all_ids  = beh_ids | demo_ids | surv_ids

cond_map = (beh_clean.dropna(subset=['study_condition'])
            .drop_duplicates('participant_id_canonical')
            .set_index('participant_id_canonical')['study_condition']
            .to_dict())

ev_summary = beh_clean.groupby('participant_id_canonical').agg(
    n_events=('event_outcome_simple', 'count'),
    n_baseline=('study_period', lambda x: (x=='baseline').sum()),
    n_intervention=('study_period', lambda x: (x=='intervention').sum()),
    n_reintervention_events=('is_reintervention', 'sum'),
    days_observed=('days_since_first_event', 'max'),
).reset_index()

linkage = pd.DataFrame({'participant_id_canonical': sorted(all_ids)})
linkage['in_behavioral']  = linkage['participant_id_canonical'].isin(beh_ids)
linkage['in_demographics'] = linkage['participant_id_canonical'].isin(demo_ids)
linkage['in_survey']       = linkage['participant_id_canonical'].isin(surv_ids)
linkage['study_condition'] = linkage['participant_id_canonical'].map(cond_map)
linkage = linkage.merge(ev_summary, on='participant_id_canonical', how='left')
linkage = linkage.merge(demo[['participant_id_canonical','age','gender','region']],
                        on='participant_id_canonical', how='left')

linkage.to_csv(OUT / "ds6_participant_linkage.csv", index=False)
print(f"Saved: ds6_participant_linkage.csv  ({linkage.shape})")
print(linkage[['in_behavioral','in_demographics','in_survey']].value_counts())

## 16. Final Summary

In [ ]:
print("""
=== DS6 CLEANING COMPLETE ===

Outputs:
  ds6_behavioral_clean.csv   : 353,796 rows × 34 cols
  ds6_survey_clean.csv       : 478 rows × 20 cols
  ds6_demographics_clean.csv : 399 rows × 6 cols
  ds6_participant_linkage.csv : 399 participants

Key notes:
  - ALL behavioral rows are openedApp — no dismiss/close in resolution
  - NO session grouping: group_id is unique per row (verified on all 353,796 rows);
    first_in_group == 1 and last_close_in_group == 0 everywhere. Kept for provenance only.
    (F197 — the earlier claim of open/close session matching was false.)
  - 243 participants have behavioral data; 156 demo-only
  - Timestamp: ISO 8601 UTC — no localization needed
  - ID = response_id — direct join, no prefix transformation

Not yet cleaned:
  - Pre-survey Qualtrics file (Danish adolescents & one sec, main study, pre-survey.csv)
  - survey_data_full.csv (raw items + 3rd wave)
""")